# ⚽ FIFA World Cup 2026 — Match Prediction System

## 📖 Background
The 2026 FIFA World Cup is one of the biggest sporting events in the world, hosted across
the United States, Canada, and Mexico. For the first time, the tournament expands to 48 teams,
producing 104 matches across the group stage and knockout rounds.

## 🧠 Approach
This prediction system is built on a **Dixon-Coles Poisson model** — a well-established
framework in football analytics that models goals scored by each team as independent Poisson
processes, with a low-score correction factor (ρ) to account for the real-world correlation
between 0-0 and 1-1 results.

The raw model is enhanced with three layers:

**1. Elo-weighted attack/defense parameters**
   Team strength is estimated from historical match data, blended with Elo ratings to
   capture recent form and tournament-level quality beyond what raw goal counts reflect.

**2. Domain adjustment layer**
   Manual overrides correct for known biases in the data — confederation strength gaps,
   Elo inflation from weak regional opposition, and first-time World Cup participants
   with limited exposure to elite competition.

**3. Deterministic bracket resolution**
   Group standings and best third-place qualifiers are derived directly from predicted
   scorelines, ensuring full internal consistency between the group stage and knockout
   bracket — every team that advances does so based on the exact scores being submitted.

## 🎯 Prediction Strategy
Scorelines are chosen to **maximise expected competition points** rather than simply
predicting the most likely scoreline. The scoring system awards:
- 25 pts for an exact scoreline
- 10 pts for correct goal difference or correct total goals
- 40 pts (group) / 20 pts (knockout) for the correct match winner

This means the optimal prediction is not always the modal scoreline — a slightly less
likely scoreline that strongly signals the correct winner can outscore a "safe" 1-1 draw.

## 📊 Model Prediction
> **Predicted Winner: 🏆 Argentina**
> **Predicted Final: Spain vs Argentina — 1:2 (a.e.t.)**

## 💾 The data

#### `data/group_fixtures.csv` — all 72 group stage matches
| Variable | Description |
|---|---|
| `match_id` | Unique match identifier |
| `group` | Group letter (A–L) |
| `home_team` | Home team name |
| `away_team` | Away team name |
| `date` | Match date (UTC) |
| `venue` | Stadium and city |

#### `data/knockout_slots.csv` — all 32 knockout round slots
| Variable | Description |
|---|---|
| `match_id` | Unique match identifier |
| `round` | Round name (e.g. `Quarter-final`) |
| `multiplier` | Score multiplier for this round |
| `slot_home` | Description of the home team slot (e.g. `Winner Group A`) |
| `slot_away` | Description of the away team slot |

### **Results and Shootout data extracted from Mart Jürisoo's International football results from 1872 to 2026**

In [38]:
import pandas as pd

group_fixtures = pd.read_csv('data/group_fixtures.csv')
group_fixtures

,match_id,group,home_team,away_team,date_utc,venue
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City"
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara"
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto"
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles"
4,5,D,Australia,UEFA Playoff C,2026-06-13T04:00:00Z,"BC Place, Vancouver"
...,...,...,...,...,...,...
67,68,L,Croatia,Ghana,2026-06-27T21:00:00Z,"Lincoln Financial Field, Philadelphia"
68,69,K,Colombia,Portugal,2026-06-27T23:30:00Z,"Hard Rock Stadium, Miami"
69,70,K,FIFA Playoff 1,Uzbekistan,2026-06-27T23:30:00Z,"Mercedes-Benz Stadium, Atlanta"
70,71,J,Algeria,Austria,2026-06-28T02:00:00Z,"GEHA Field at Arrowhead Stadium, Kansas City"


In [39]:
knockout_slots = pd.read_csv('data/knockout_slots.csv')
knockout_slots

,match_id,round,multiplier,date_utc,venue,slot_home,slot_away
0,73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B
1,74,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F
2,75,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F)
3,76,Round of 32,1,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C
4,77,Round of 32,1,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I
5,78,Round of 32,1,2026-06-30T21:00:00Z,"MetLife Stadium, East Rutherford",Winner Group I,Best 3rd (Groups C/D/F/G/H)
6,79,Round of 32,1,2026-07-01T01:00:00Z,"Estadio Azteca, Mexico City",Winner Group A,Best 3rd (Groups C/E/F/H/I)
7,80,Round of 32,1,2026-07-01T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Group L,Best 3rd (Groups E/H/I/J/K)
8,81,Round of 32,1,2026-07-01T20:00:00Z,"Lumen Field, Seattle",Winner Group G,Best 3rd (Groups A/E/H/I/J)
9,82,Round of 32,1,2026-07-02T00:00:00Z,"Levi's Stadium, Santa Clara",Winner Group D,Best 3rd (Groups B/E/F/I/J)


## Replace playoff placeholders with actual qualified teams

In [40]:
replacements = {
    'UEFA Playoff A': 'Bosnia & Herzegovina',
    'UEFA Playoff B': 'Sweden',
    'UEFA Playoff C': 'Türkiye',
    'UEFA Playoff D': 'Czechia',
    'FIFA Playoff 1': 'DR Congo',
    'FIFA Playoff 2': 'Iraq',
}

for placeholder, actual_team in replacements.items():
    group_fixtures.loc[group_fixtures['home_team'] == placeholder, 'home_team'] = actual_team
    group_fixtures.loc[group_fixtures['away_team'] == placeholder, 'away_team'] = actual_team

# Verify no placeholders remain
remaining = group_fixtures[
    group_fixtures['home_team'].str.contains('UEFA|FIFA', na=False) |
    group_fixtures['away_team'].str.contains('UEFA|FIFA', na=False)
]
print(f"Remaining placeholders: {len(remaining)}")  # Should be 0
group_fixtures.head(10)

Remaining placeholders: 0


,match_id,group,home_team,away_team,date_utc,venue
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City"
1,2,A,South Korea,Czechia,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara"
2,3,B,Canada,Bosnia & Herzegovina,2026-06-12T19:00:00Z,"BMO Field, Toronto"
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles"
4,5,D,Australia,Türkiye,2026-06-13T04:00:00Z,"BC Place, Vancouver"
5,6,B,Qatar,Switzerland,2026-06-13T19:00:00Z,"Levi's Stadium, Santa Clara"
6,7,C,Brazil,Morocco,2026-06-13T22:00:00Z,"MetLife Stadium, East Rutherford"
7,8,C,Haiti,Scotland,2026-06-14T01:00:00Z,"Gillette Stadium, Boston"
8,9,E,Germany,Curaçao,2026-06-14T17:00:00Z,"NRG Stadium, Houston"
9,10,F,Netherlands,Japan,2026-06-14T20:00:00Z,"AT&T Stadium, Dallas"


## Import & fetch the raw data

In [41]:
import pandas as pd
import numpy as np

RESULTS_URL   = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"
SHOOTOUTS_URL = "https://raw.githubusercontent.com/martj42/international_results/master/shootouts.csv"

SNAPSHOT_DATE = '2026-06-09'  # freeze data before the deadline

results   = pd.read_csv(RESULTS_URL,   parse_dates=['date'])
shootouts = pd.read_csv(SHOOTOUTS_URL, parse_dates=['date'])

# Apply snapshot filter immediately after loading
results   = results[results['date'] <= SNAPSHOT_DATE].reset_index(drop=True)
shootouts = shootouts[shootouts['date'] <= SNAPSHOT_DATE].reset_index(drop=True)

print(f"Results:   {len(results):,} matches  ({results['date'].min().date()} → {results['date'].max().date()})")
print(f"Shootouts: {len(shootouts):,} matches")
results.head(3)

Results:   49,373 matches  (1872-11-30 → 2026-06-07)
Shootouts: 678 matches


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False


## Remap team names to match fixture names

In [42]:
NAME_MAP = {
    'Czech Republic':         'Czechia',
    'Bosnia and Herzegovina': 'Bosnia & Herzegovina',
    'United States':          'USA',
    'Turkey':                 'Türkiye',
    'Ivory Coast':            "Côte d'Ivoire",
    'Cape Verde':             'Cabo Verde',
}
 
results['home_team'] = results['home_team'].replace(NAME_MAP)
results['away_team'] = results['away_team'].replace(NAME_MAP)
 
OUR_TEAMS = [
    'Mexico', 'South Africa', 'South Korea', 'Czechia',
    'Canada', 'Qatar', 'Switzerland', 'Bosnia & Herzegovina',
    'Brazil', 'Morocco', 'Haiti', 'Scotland',
    'USA', 'Paraguay', 'Australia', 'Türkiye',
    'Germany', "Côte d'Ivoire", 'Ecuador', 'Curaçao',
    'Netherlands', 'Japan', 'Tunisia', 'Sweden',
    'Belgium', 'Egypt', 'Iran', 'New Zealand',
    'Spain', 'Saudi Arabia', 'Uruguay', 'Cabo Verde',
    'France', 'Senegal', 'Norway', 'Iraq',
    'Argentina', 'Austria', 'Jordan', 'Algeria',
    'Portugal', 'Colombia', 'Uzbekistan', 'DR Congo',
    'England', 'Croatia', 'Ghana', 'Panama',
]
 
all_in_data = set(results['home_team']) | set(results['away_team'])
missing = [t for t in OUR_TEAMS if t not in all_in_data]
print(f"Teams missing from dataset after remapping: {missing if missing else 'None — all 48 found ✓'}")

Teams missing from dataset after remapping: None — all 48 found ✓


## Filter to recent matches (post-2022 WC cycle)

In [43]:
CUTOFF = '2022-01-01'
 
recent = results[
    (results['date'] >= CUTOFF) &
    (
        results['home_team'].isin(OUR_TEAMS) |
        results['away_team'].isin(OUR_TEAMS)
    )
].copy().reset_index(drop=True)
 
print(f"Recent matches (since {CUTOFF}): {len(recent):,}")
print(f"\nTop tournament types:")
print(recent['tournament'].value_counts().head(10).to_string())

Recent matches (since 2022-01-01): 2,051

Top tournament types:
tournament
Friendly                                592
FIFA World Cup qualification            532
UEFA Nations League                     174
African Cup of Nations qualification    112
UEFA Euro qualification                 109
African Cup of Nations                  105
CONCACAF Nations League                  75
FIFA World Cup                           63
Gold Cup                                 45
UEFA Euro                                44


## Compute Elo ratings from full match history

In [44]:
def compute_elo_ratings(results_df, k_competitive=40, k_friendly=20, base=1500):
    """
    Replay match history chronologically to compute Elo ratings.
    K=40 for competitive matches, K=20 for friendlies.
    Returns dict: {team: elo}
    """
    elo = {}
    friendly_tournaments = {'Friendly'}
 
    for _, row in results_df.sort_values('date').iterrows():
        home, away = row['home_team'], row['away_team']
        hs, as_ = row['home_score'], row['away_score']
 
        if pd.isna(hs) or pd.isna(as_):
            continue
 
        if home not in elo: elo[home] = base
        if away not in elo: elo[away] = base
 
        exp_home = 1 / (1 + 10 ** ((elo[away] - elo[home]) / 400))
        exp_away = 1 - exp_home
 
        if   hs > as_: act_home, act_away = 1.0, 0.0
        elif hs < as_: act_home, act_away = 0.0, 1.0
        else:          act_home, act_away = 0.5, 0.5
 
        k = k_friendly if row['tournament'] in friendly_tournaments else k_competitive
 
        elo[home] += k * (act_home - exp_home)
        elo[away] += k * (act_away - exp_away)
 
    return elo
 
elo_ratings = compute_elo_ratings(results)
 
elo_df = (
    pd.DataFrame.from_dict(elo_ratings, orient='index', columns=['elo'])
    .reset_index()
    .rename(columns={'index': 'team'})
    .query("team in @OUR_TEAMS")
    .sort_values('elo', ascending=False)
    .reset_index(drop=True)
)
 
print(f"Elo computed for {len(elo_df)} / 48 teams\n")
print(elo_df.to_string(index=False))

Elo computed for 48 / 48 teams

                team         elo
               Spain 2074.806548
           Argentina 2046.213348
              France 1998.803638
             England 1965.599941
            Portugal 1949.894054
             Germany 1941.308245
              Brazil 1934.108750
               Japan 1928.556528
             Morocco 1927.509129
            Colombia 1926.952469
         Netherlands 1915.038543
             Ecuador 1912.906642
              Mexico 1895.700360
             Croatia 1889.721290
             Türkiye 1888.186934
             Uruguay 1866.798485
              Norway 1861.667986
             Belgium 1858.667891
         Switzerland 1852.126794
           Australia 1843.115932
         South Korea 1838.187119
                Iran 1837.432546
             Senegal 1832.286181
              Canada 1826.487071
                 USA 1825.787222
            Paraguay 1825.187595
             Algeria 1815.990622
             Austria 1814.565384
          U

## Compute attack & defense strength from recent matches

In [45]:
def compute_attack_defense(recent_df, our_teams, min_matches=3):
    """
    attack_strength  = avg goals scored per game
    defense_strength = avg goals conceded per game
    Falls back to global average for teams with < min_matches games.
    """
    records = []
    for team in our_teams:
        home_games = recent_df[recent_df['home_team'] == team]
        away_games = recent_df[recent_df['away_team'] == team]
 
        goals_scored   = home_games['home_score'].sum() + away_games['away_score'].sum()
        goals_conceded = home_games['away_score'].sum() + away_games['home_score'].sum()
        n_games        = len(home_games) + len(away_games)
 
        records.append({
            'team':             team,
            'goals_scored':     goals_scored,
            'goals_conceded':   goals_conceded,
            'n_recent_games':   n_games,
            'attack_strength':  round(goals_scored   / n_games, 3) if n_games >= min_matches else None,
            'defense_strength': round(goals_conceded / n_games, 3) if n_games >= min_matches else None,
        })
 
    df = pd.DataFrame(records)
 
    # Fill sparse teams with global average
    global_avg_attack  = df['attack_strength'].mean()
    global_avg_defense = df['defense_strength'].mean()
    df['attack_strength']  = df['attack_strength'].fillna(global_avg_attack)
    df['defense_strength'] = df['defense_strength'].fillna(global_avg_defense)
 
    sparse = df[df['n_recent_games'] < min_matches]['team'].tolist()
    if sparse:
        print(f"Teams using global average fallback (< {min_matches} recent games): {sparse}")
 
    return df
 
attack_defense = compute_attack_defense(recent, OUR_TEAMS)

## Build master team_strength table

In [46]:
GROUPS = {
    'A': ['Mexico', 'South Africa', 'South Korea', 'Czechia'],
    'B': ['Canada', 'Qatar', 'Switzerland', 'Bosnia & Herzegovina'],
    'C': ['Brazil', 'Morocco', 'Haiti', 'Scotland'],
    'D': ['USA', 'Paraguay', 'Australia', 'Türkiye'],
    'E': ['Germany', "Côte d'Ivoire", 'Ecuador', 'Curaçao'],
    'F': ['Netherlands', 'Japan', 'Tunisia', 'Sweden'],
    'G': ['Belgium', 'Egypt', 'Iran', 'New Zealand'],
    'H': ['Spain', 'Saudi Arabia', 'Uruguay', 'Cabo Verde'],
    'I': ['France', 'Senegal', 'Norway', 'Iraq'],
    'J': ['Argentina', 'Austria', 'Jordan', 'Algeria'],
    'K': ['Portugal', 'Colombia', 'Uzbekistan', 'DR Congo'],
    'L': ['England', 'Croatia', 'Ghana', 'Panama'],
}
 
group_map = {team: grp for grp, teams in GROUPS.items() for team in teams}
 
team_strength = (
    elo_df
    .merge(attack_defense[['team', 'attack_strength', 'defense_strength', 'n_recent_games']], on='team')
    .assign(group=lambda df: df['team'].map(group_map))
    [['team', 'group', 'elo', 'attack_strength', 'defense_strength', 'n_recent_games']]
    .sort_values('elo', ascending=False)
    .reset_index(drop=True)
)
 
print("=== Master Team Strength Table ===")
print(team_strength.to_string(index=False))

=== Master Team Strength Table ===
                team group         elo  attack_strength  defense_strength  n_recent_games
               Spain     H 2074.806548            2.491             0.811              53
           Argentina     J 2046.213348            2.278             0.444              54
              France     I 1998.803638            2.111             0.926              54
             England     L 1965.599941            1.981             0.717              53
            Portugal     K 1949.894054            2.491             0.792              53
             Germany     E 1941.308245            2.058             1.212              52
              Brazil     C 1934.108750            2.020             0.960              50
               Japan     F 1928.556528            2.554             0.696              56
             Morocco     C 1927.509129            1.944             0.493              71
            Colombia     K 1926.952469            1.843          

One thing to flag from the data: Haiti's attack_strength = 2.22 looks suspiciously high — likely inflated by goals against weak CONCACAF opposition. Worth keeping in mind teams like that when applying domain adjustments later.

## Prepare Dixon-Coles training data

In [47]:
# Requires: results, OUR_TEAMS  (from data collection cells)
 
from scipy.optimize import minimize
from scipy.stats import poisson
import warnings
warnings.filterwarnings('ignore')
 
CUTOFF_DC = '2018-01-01'   # ~2 WC cycles of history; enough data, not too stale
 
dc_data = results[
    (results['date'] >= CUTOFF_DC) &
    results['home_team'].isin(OUR_TEAMS) &
    results['away_team'].isin(OUR_TEAMS) &
    results['home_score'].notna() &
    results['away_score'].notna()
].copy().reset_index(drop=True)
 
# Time-decay weights: recent matches count more (half-life ~1 year)
max_date            = dc_data['date'].max()
dc_data['days_ago'] = (max_date - dc_data['date']).dt.days
dc_data['weight']   = np.exp(-dc_data['days_ago'] / 365)
 
print(f"D-C training matches : {len(dc_data):,}")
print(f"Date range           : {dc_data['date'].min().date()} -> {dc_data['date'].max().date()}")
print(f"Avg match weight     : {dc_data['weight'].mean():.3f}  (1.0=today, 0.37=1yr ago)")

D-C training matches : 997
Date range           : 2018-01-28 -> 2026-06-07
Avg match weight     : 0.131  (1.0=today, 0.37=1yr ago)


## Fit Dixon-Coles model

In [48]:
teams    = sorted(OUR_TEAMS)
n_teams  = len(teams)
team_idx = {t: i for i, t in enumerate(teams)}

# --- Tournament weighting (NEW): folded into the existing recency weight ---
def tournament_weight(t):
    t = str(t).lower()
    if 'friendly' in t:                                          return 0.7
    if 'nations league' in t:                                    return 1.1
    if 'qualif' in t:                                            return 1.2
    if any(k in t for k in ['world cup', 'confederations']):     return 1.5
    if any(k in t for k in ['euro','copa am','african cup',
                            'asian cup','gold cup','nations cup']): return 1.3
    return 1.0

dc_data['w_tourney'] = dc_data['tournament'].map(tournament_weight)
dc_data['weight']    = dc_data['weight'] * dc_data['w_tourney']     # recency * tourney
neutral_i = dc_data['neutral'].astype(int).values if 'neutral' in dc_data.columns else np.zeros(len(dc_data), int)
not_neutral = 1 - neutral_i

home_idx   = dc_data['home_team'].map(team_idx).values
away_idx   = dc_data['away_team'].map(team_idx).values
home_goals = dc_data['home_score'].astype(int).values
away_goals = dc_data['away_score'].astype(int).values
weights    = dc_data['weight'].values

SHRINK = 0.0   # ridge shrinkage. KEEP AT 0: tested, it drags genuinely-weak but
               # well-sampled teams (Qatar) toward average. The Elo blend below
               # handles thin-sample teams better. Knob left for experimentation.

def neg_log_likelihood(params):
    attack   = params[:n_teams]
    defense  = params[n_teams:2*n_teams]
    home_adv = params[2*n_teams]
    rho      = params[2*n_teams + 1]
    # home_adv applies ONLY to genuine (non-neutral) home matches
    lambda_h = np.clip(np.exp(attack[home_idx] + defense[away_idx] + home_adv*not_neutral), 1e-6, None)
    lambda_a = np.clip(np.exp(attack[away_idx] + defense[home_idx]),                          1e-6, None)
    tau = np.ones(len(dc_data))
    m00=(home_goals==0)&(away_goals==0); m01=(home_goals==0)&(away_goals==1)
    m10=(home_goals==1)&(away_goals==0); m11=(home_goals==1)&(away_goals==1)
    tau[m00]=np.clip(1-lambda_h[m00]*lambda_a[m00]*rho,1e-6,None)
    tau[m01]=np.clip(1+lambda_h[m01]*rho,1e-6,None)
    tau[m10]=np.clip(1+lambda_a[m10]*rho,1e-6,None)
    tau[m11]=np.clip(1-rho,1e-6,None)
    ll = (weights*(np.log(tau)+poisson.logpmf(home_goals,lambda_h)+poisson.logpmf(away_goals,lambda_a))).sum()
    return -ll + SHRINK*(np.sum(attack**2)+np.sum(defense**2))

x0 = np.zeros(2*n_teams + 2); x0[2*n_teams]=0.1; x0[2*n_teams+1]=-0.1
print("Fitting Dixon-Coles (tournament-weighted, neutral-aware)...")
dc_result = minimize(neg_log_likelihood, x0, method='SLSQP',
                     constraints=[{'type':'eq','fun':lambda x: np.sum(x[:n_teams])}],
                     options={'maxiter':500,'ftol':1e-9})
fitted = dc_result.x
_raw_attack  = dict(zip(teams, fitted[:n_teams]))
_raw_defense = dict(zip(teams, fitted[n_teams:2*n_teams]))
dc_home_adv  = fitted[2*n_teams]      # true competitive home adv (~0.36); WC uses host bump only
dc_rho       = fitted[2*n_teams + 1]
print(f"Converged: {dc_result.success}   home_adv={dc_home_adv:.3f} (competitive only)   rho={dc_rho:.3f}")

# --- Elo blend (NEW): reconcile goals-based DC with result-based Elo on OVERALL quality ---
ELO_BLEND = 0.50   # weight on Elo. 0.30 trusts goals more; 0.60 trusts Elo more
                   # (0.60 puts Türkiye in the Group D top-2 fight, matching Elo+odds).
_overall = {t: _raw_attack[t] - _raw_defense[t] for t in teams}     # higher = better
_ov = np.array([_overall[t] for t in teams]); _ovm, _ovs = _ov.mean(), _ov.std()
_el = np.array([elo_ratings[t] for t in teams]); _elm, _els = _el.mean(), _el.std()

dc_attack, dc_defense = {}, {}
for t in teams:
    blended_z = (1-ELO_BLEND)*((_overall[t]-_ovm)/_ovs) + ELO_BLEND*((elo_ratings[t]-_elm)/_els)
    target    = _ovm + blended_z*_ovs
    delta     = target - _overall[t]
    dc_attack[t]  = _raw_attack[t]  + delta/2      # split the shift, preserve attack/defense style
    dc_defense[t] = _raw_defense[t] - delta/2

print(f"Elo blend applied (w={ELO_BLEND}). Final dc_attack/dc_defense ready.")

Fitting Dixon-Coles (tournament-weighted, neutral-aware)...
Converged: True   home_adv=0.159 (competitive only)   rho=-0.139
Elo blend applied (w=0.5). Final dc_attack/dc_defense ready.


## Inspect fitted parameters

In [49]:
dc_params_df = (
    pd.DataFrame({'team': teams,
                  'attack':  [dc_attack[t]  for t in teams],
                  'defense': [dc_defense[t] for t in teams]})
    .sort_values('attack', ascending=False)
    .reset_index(drop=True)
)
 
print("=== Top 10 Attack (higher = more goals scored) ===")
print(dc_params_df[['team','attack']].head(10).to_string(index=False))
 
print("\n=== Top 10 Defense (lower = harder to score against) ===")
print(dc_params_df[['team','defense']].sort_values('defense').head(10).to_string(index=False))

=== Top 10 Attack (higher = more goals scored) ===
       team   attack
      Spain 0.914100
     Brazil 0.689816
    Belgium 0.665796
     France 0.612465
    Germany 0.605428
  Argentina 0.558106
   Colombia 0.511586
Netherlands 0.483011
    Morocco 0.449014
Switzerland 0.436996

=== Top 10 Defense (lower = harder to score against) ===
     team   defense
  Ecuador -0.866013
Argentina -0.600861
 Portugal -0.597556
  Algeria -0.573478
   Canada -0.423065
   France -0.373735
  England -0.340446
   Mexico -0.308317
    Japan -0.305000
  Uruguay -0.187373


## Core prediction functions

In [50]:
LAMBDA_CAP = 6.0   # realistic ceiling; prevents extreme blowout inflation
 
def predict_match(home_team, away_team, neutral=False, max_goals=8):
    """
    Returns scoreline probability matrix + win/draw/loss probabilities.
 
    Parameters
    ----------
    home_team, away_team : str
    neutral : bool  — True for knockout matches (no home advantage)
    max_goals : int — truncate distribution at this many goals per side
 
    Returns dict with keys:
      score_matrix, lambda_h, lambda_a, p_home_win, p_draw, p_away_win
    """
    adv      = 0 if neutral else dc_home_adv
    lambda_h = min(np.exp(dc_attack[home_team] + dc_defense[away_team] + adv), LAMBDA_CAP)
    lambda_a = min(np.exp(dc_attack[away_team] + dc_defense[home_team]),       LAMBDA_CAP)
 
    h_probs = poisson.pmf(np.arange(max_goals + 1), lambda_h)
    a_probs = poisson.pmf(np.arange(max_goals + 1), lambda_a)
    matrix  = np.outer(h_probs, a_probs)
 
    # Dixon-Coles low-score correction
    matrix[0, 0] *= max(1 - lambda_h * lambda_a * dc_rho, 1e-6)
    matrix[0, 1] *= max(1 + lambda_h * dc_rho,            1e-6)
    matrix[1, 0] *= max(1 + lambda_a * dc_rho,            1e-6)
    matrix[1, 1] *= max(1 - dc_rho,                       1e-6)
    matrix /= matrix.sum()
 
    return {
        'score_matrix': matrix,
        'lambda_h':     lambda_h,
        'lambda_a':     lambda_a,
        'p_home_win':   float(np.tril(matrix, -1).sum()),
        'p_draw':       float(np.trace(matrix)),
        'p_away_win':   float(np.triu(matrix,  1).sum()),
    }
 
 
def most_likely_score(pred):
    """Single most probable scoreline (home_goals, away_goals)."""
    idx = np.unravel_index(pred['score_matrix'].argmax(), pred['score_matrix'].shape)
    return int(idx[0]), int(idx[1])
 
 
def best_scoreline_for_points(pred, include_winner_bonus=False, winner_bonus_pts=40):
    """
    Scoreline that maximises expected competition points.
      Exact scoreline        -> 25 pts
      Correct goal diff only -> 10 pts
      Correct total goals    -> 10 pts
      Correct winning team   -> winner_bonus_pts (40 group stage, 20 KO)
    """
    matrix = pred['score_matrix']
    mg     = matrix.shape[0]
    best_ep, best_pred = -1, (1, 1)

    for ph in range(mg):
        for pa in range(mg):
            ep = 0.0
            pred_result = 'H' if ph > pa else ('A' if ph < pa else 'D')
            for ah in range(mg):
                for aa in range(mg):
                    p = matrix[ah, aa]
                    if ph == ah and pa == aa:
                        ep += p * 25
                    elif (ph - pa) == (ah - aa):
                        ep += p * 10
                    elif (ph + pa) == (ah + aa):
                        ep += p * 10
                    if include_winner_bonus:
                        act_result = 'H' if ah > aa else ('A' if ah < aa else 'D')
                        if pred_result == act_result:
                            ep += p * winner_bonus_pts
            if ep > best_ep:
                best_ep, best_pred = ep, (ph, pa)

    return best_pred, best_ep

## Domain adjustment layer

In [51]:

# ── MANUAL OVERRIDES ──────────────────────────────────────────────────────────
#
# Improvement 1 (Group D): Türkiye has the highest Elo
#   in Group D (1881) and reached the Euro 2024 knockout rounds.
#
# Improvement 2 (Group J): Algeria edges Austria in the raw model, but Austria
#   reached the Euro 2024 quarter-finals (their best ever) while Algeria's strength
#   comes mainly from AFCON competition. Competition-quality context matters.
#
# Improvement 3 (Group I): Norway's raw attack (2.326) is Haaland-inflated from
#   Nations League games vs weaker sides. They failed to qualify for Euro 2024.
#
MANUAL_OVERRIDES = {
    'Australia': {'attack': 0.88, 'defense': 1.06},  # 2022 run inflates Elo; weak AFC path
    'USA':       {'attack': 1.08, 'defense': 0.95},  # hosts
    'Türkiye':  {'attack': 1.12, 'defense': 0.95},   # Euro 2024 performer; Elo says best in D

    # Group I fix
    'Norway':   {'attack': 0.90},                    # Small adjustment

    # Group J fix
    'Austria':  {'attack': 1.10, 'defense': 0.92},   # Euro 2024 QF
    'Algeria':  {'defense': 1.2},                   # mostly AFCON; slight debuff vs EUR quality
}

import copy
dc_attack_adj  = copy.deepcopy(dc_attack)
dc_defense_adj = copy.deepcopy(dc_defense)
for tm, adj in MANUAL_OVERRIDES.items():
    if 'attack'  in adj: dc_attack_adj[tm]  += np.log(adj['attack'])
    if 'defense' in adj: dc_defense_adj[tm] += np.log(adj['defense'])

HOSTS     = {'USA', 'Canada', 'Mexico'}
HOST_ADV  = 0.06
LAMBDA_CAP = 6.0

def predict_match_adj(home_team, away_team, neutral=True, max_goals=8):
    adv_h = HOST_ADV if home_team in HOSTS else 0.0
    adv_a = HOST_ADV if away_team in HOSTS else 0.0
    lambda_h = min(np.exp(dc_attack_adj[home_team] + dc_defense_adj[away_team] + adv_h), LAMBDA_CAP)
    lambda_a = min(np.exp(dc_attack_adj[away_team] + dc_defense_adj[home_team] + adv_a), LAMBDA_CAP)
    h_probs = poisson.pmf(np.arange(max_goals+1), lambda_h)
    a_probs = poisson.pmf(np.arange(max_goals+1), lambda_a)
    matrix  = np.outer(h_probs, a_probs)
    matrix[0,0] *= max(1-lambda_h*lambda_a*dc_rho,1e-6)
    matrix[0,1] *= max(1+lambda_h*dc_rho,1e-6)
    matrix[1,0] *= max(1+lambda_a*dc_rho,1e-6)
    matrix[1,1] *= max(1-dc_rho,1e-6)
    matrix /= matrix.sum()
    return {'score_matrix':matrix,'lambda_h':lambda_h,'lambda_a':lambda_a,
            'p_home_win':float(np.tril(matrix,-1).sum()),
            'p_draw':float(np.trace(matrix)),
            'p_away_win':float(np.triu(matrix,1).sum())}

print(f"Predictor ready. ELO_BLEND={ELO_BLEND}, HOST_ADV={HOST_ADV}")
print(f"Manual overrides applied: {list(MANUAL_OVERRIDES.keys())}")
print()
print("=== Quick sanity: Group D (Paraguay issue) ===")
for h, a in [('USA','Paraguay'), ('Türkiye','Paraguay'), ('Türkiye','USA'), ('Türkiye','Australia')]:
    p = predict_match_adj(h, a, neutral=False)
    print(f"  {h} vs {a}: {p['p_home_win']:.1%} / {p['p_draw']:.1%} / {p['p_away_win']:.1%}")
print()
print("=== Quick sanity: Group J (Algeria vs Austria) ===")
p = predict_match_adj('Algeria','Austria', neutral=False)
print(f"  Algeria vs Austria: {p['p_home_win']:.1%} / {p['p_draw']:.1%} / {p['p_away_win']:.1%}")


Predictor ready. ELO_BLEND=0.5, HOST_ADV=0.06
Manual overrides applied: ['Australia', 'USA', 'Türkiye', 'Norway', 'Austria', 'Algeria']

=== Quick sanity: Group D (Paraguay issue) ===
  USA vs Paraguay: 33.6% / 30.0% / 36.4%
  Türkiye vs Paraguay: 36.1% / 30.2% / 33.6%
  Türkiye vs USA: 40.9% / 24.5% / 34.6%
  Türkiye vs Australia: 42.4% / 30.3% / 27.3%

=== Quick sanity: Group J (Algeria vs Austria) ===
  Algeria vs Austria: 28.8% / 37.4% / 33.7%


## Monte Carlo group stage simulation
Run the cells below to simulate the group stage 50,000 times, deriving each team's
probability of finishing 1st, 2nd, 3rd, or 4th in their group. These probabilities
inform the expected finishing position used to determine the most likely group standings
and best third-place qualifiers that feed into the knockout bracket.

> ⚠️ **Note:** This section is currently disabled — group standings and best third-place
> qualifiers are derived directly from the predicted scorelines for full consistency
> with the submission. Re-enable if you want probabilistic finish percentages for analysis

In [52]:
# N_SIMULATIONS = 50_000

# GROUPS = {
#     'A': ['Mexico', 'South Africa', 'South Korea', 'Czechia'],
#     'B': ['Canada', 'Qatar', 'Switzerland', 'Bosnia & Herzegovina'],
#     'C': ['Brazil', 'Morocco', 'Haiti', 'Scotland'],
#     'D': ['USA', 'Paraguay', 'Australia', 'Türkiye'],
#     'E': ['Germany', "Côte d'Ivoire", 'Ecuador', 'Curaçao'],
#     'F': ['Netherlands', 'Japan', 'Tunisia', 'Sweden'],
#     'G': ['Belgium', 'Egypt', 'Iran', 'New Zealand'],
#     'H': ['Spain', 'Saudi Arabia', 'Uruguay', 'Cabo Verde'],
#     'I': ['France', 'Senegal', 'Norway', 'Iraq'],
#     'J': ['Argentina', 'Austria', 'Jordan', 'Algeria'],
#     'K': ['Portugal', 'Colombia', 'Uzbekistan', 'DR Congo'],
#     'L': ['England', 'Croatia', 'Ghana', 'Panama'],
# }

# group_fixtures_list = []
# for grp, teams in GROUPS.items():
#     for i in range(len(teams)):
#         for j in range(i+1, len(teams)):
#             group_fixtures_list.append((grp, teams[i], teams[j]))


# def simulate_group_once(teams, rng):
#     """
#     Simulate one full group round-robin.
#     Returns standings dict: {team: {pts, gf, ga, gd}}
#     """
#     s = {t: {'pts': 0, 'gf': 0, 'ga': 0, 'gd': 0} for t in teams}
#     matchups = [(teams[i], teams[j])
#                 for i in range(len(teams))
#                 for j in range(i+1, len(teams))]

#     for home, away in matchups:
#         pred   = predict_match_adj(home, away, neutral=False)
#         matrix = pred['score_matrix']
#         flat   = matrix.flatten()
#         flat  /= flat.sum()
#         idx    = rng.choice(len(flat), p=flat)
#         hg, ag = divmod(idx, matrix.shape[1])

#         s[home]['gf'] += hg;  s[home]['ga'] += ag
#         s[away]['gf'] += ag;  s[away]['ga'] += hg
#         s[home]['gd']  = s[home]['gf'] - s[home]['ga']
#         s[away]['gd']  = s[away]['gf'] - s[away]['ga']

#         if   hg > ag: s[home]['pts'] += 3
#         elif hg < ag: s[away]['pts'] += 3
#         else:         s[home]['pts'] += 1; s[away]['pts'] += 1

#     return s


# def rank_group(standings, teams, rng):
#     """
#     Rank teams by pts → gd → gf → random (for ties).
#     Returns ordered list [1st, 2nd, 3rd, 4th].
#     """
#     def sort_key(t):
#         return (
#             -standings[t]['pts'],
#             -standings[t]['gd'],
#             -standings[t]['gf'],
#             rng.random()          # random tiebreaker (avoids bias)
#         )
#     return sorted(teams, key=sort_key)


# print(f"Running {N_SIMULATIONS:,} Monte Carlo group simulations...")

# rng = np.random.default_rng(seed=42)

# # Accumulators
# finish_counts = {grp: {t: {pos: 0 for pos in [1,2,3,4]}
#                         for t in GROUPS[grp]}
#                  for grp in GROUPS}

# third_place_records = []   # list of (team, pts, gd, gf) per simulation

# for sim in range(N_SIMULATIONS):
#     sim_thirds = []

#     for grp, teams in GROUPS.items():
#         standings = simulate_group_once(teams, rng)
#         ranked    = rank_group(standings, teams, rng)

#         for pos, team in enumerate(ranked, 1):
#             finish_counts[grp][team][pos] += 1

#         # Record 3rd place finisher stats for best-3rd logic
#         third_team = ranked[2]
#         sim_thirds.append({
#             'group': grp,
#             'team':  third_team,
#             'pts':   standings[third_team]['pts'],
#             'gd':    standings[third_team]['gd'],
#             'gf':    standings[third_team]['gf'],
#         })

#     # Rank all 12 third-place teams; top 8 qualify
#     sim_thirds_sorted = sorted(
#         sim_thirds,
#         key=lambda x: (-x['pts'], -x['gd'], -x['gf'], rng.random())
#     )
#     for rank, rec in enumerate(sim_thirds_sorted):
#         rec['third_rank'] = rank + 1   # 1=best, 12=worst

#     third_place_records.append(sim_thirds_sorted)

# print("Simulation complete.")

In [53]:
# # Extract qualification probabilities
# # --- Group finish probabilities ---
# finish_probs = {}
# for grp in GROUPS:
#     finish_probs[grp] = {}
#     for team in GROUPS[grp]:
#         finish_probs[grp][team] = {
#             pos: finish_counts[grp][team][pos] / N_SIMULATIONS
#             for pos in [1,2,3,4]
#         }

# print("=== Group Qualification Probabilities ===\n")
# for grp in sorted(GROUPS.keys()):
#     print(f"Group {grp}:")
#     teams_sorted = sorted(
#         GROUPS[grp],
#         key=lambda t: -finish_probs[grp][t][1]
#     )
#     print(f"  {'Team':<25} {'1st':>7} {'2nd':>7} {'3rd':>7} {'4th':>7}")
#     for team in teams_sorted:
#         p = finish_probs[grp][team]
#         print(f"  {team:<25} {p[1]:>7.1%} {p[2]:>7.1%} {p[3]:>7.1%} {p[4]:>7.1%}")
#     print()


# # --- Best 3rd place qualification probability ---
# third_qualify_counts = {}
# for sim_thirds in third_place_records:
#     for rec in sim_thirds:
#         if rec['third_rank'] <= 8:   # top 8 third-place teams qualify
#             team = rec['team']
#             third_qualify_counts[team] = third_qualify_counts.get(team, 0) + 1

# print("=== Best 3rd Place Qualification Probability ===")
# third_qual_probs = {
#     t: third_qualify_counts.get(t, 0) / N_SIMULATIONS
#     for grp in GROUPS for t in GROUPS[grp]
# }
# # Show only teams with meaningful probability
# third_candidates = sorted(
#     [(t, p) for t, p in third_qual_probs.items() if p > 0.01],
#     key=lambda x: -x[1]
# )
# for team, prob in third_candidates:
#     grp = next(g for g, ts in GROUPS.items() if team in ts)
#     print(f"  {team:<25} (Group {grp})  {prob:.1%}")

## 🗓️ Group stage predictions

In [54]:
# ── GROUP STAGE PREDICTIONS ───────────────────────────────────────────────────
# Score   : best_scoreline_for_points — maximises expected competition points
# Winner  : model win-probability (independent of score; scored separately)
# Corners : scaled from expected total goals using WC 2022 baseline (~10.3/game)
# Yellows : 3 baseline; 4 for heavy mismatches (frustrated underdog effect)
# Reds    : 0 (modal outcome; ~0.17/game at WC so never predict 1)
# ─────────────────────────────────────────────────────────────────────────────

group_predictions = group_fixtures.copy()
rows = []

for _, match in group_fixtures.iterrows():
    home = match['home_team']
    away = match['away_team']

    pred = predict_match_adj(home, away, neutral=False)
    (hg, ag), _ = best_scoreline_for_points(pred, include_winner_bonus=True)

    # Winning team — derived from the scoreline we predicted (consistent with score)
    if hg > ag:
        winner = 'home'
    elif ag > hg:
        winner = 'away'
    else:
        winner = 'draw'

    # Corners: WC 2022 avg = 10.3; scale linearly with total expected goals
    # avg total goals in WC group stage ≈ 2.65
    lam_total = pred['lambda_h'] + pred['lambda_a']
    corners   = max(7, min(14, round(10.3 * lam_total / 2.65)))

    # Yellow cards: 4 for big mismatches (underdogs foul more), else 3
    gap = abs(pred['p_home_win'] - pred['p_away_win'])
    yellows = 4 if gap > 0.45 else 3

    rows.append({
        'predicted_home_goals': int(hg),
        'predicted_away_goals': int(ag),
        'corners':              int(corners),
        'yellow_cards':         int(yellows),
        'red_cards':            0,
        'winning_team':         winner,
    })

for col in ['predicted_home_goals','predicted_away_goals','corners',
            'yellow_cards','red_cards','winning_team']:
    group_predictions[col] = [r[col] for r in rows]

print(f"Group predictions filled: {len(rows)} matches")
print()
print(group_predictions[['match_id','group','home_team','away_team',
                          'predicted_home_goals','predicted_away_goals',
                          'corners','yellow_cards','red_cards',
                          'winning_team']].to_string(index=False))


Group predictions filled: 72 matches

 match_id group            home_team            away_team  predicted_home_goals  predicted_away_goals  corners  yellow_cards  red_cards winning_team
        1     A               Mexico         South Africa                     1                     0        7             4          0         home
        2     A          South Korea              Czechia                     2                     1       10             3          0         home
        3     B               Canada Bosnia & Herzegovina                     1                     0        7             3          0         home
        4     D                  USA             Paraguay                     1                     2       10             3          0         away
        5     D            Australia              Türkiye                     1                     2       10             3          0         away
        6     B                Qatar          Switzerland           

In [55]:
# ── GROUP STAGE PREDICTIONS ───────────────────────────────────────────────────
# ── Display ───────────────────────────────────────────────────────────────────
from IPython.display import display

group_predictions['scoreline'] = (
    group_predictions['predicted_home_goals'].astype(str) + ' - ' +
    group_predictions['predicted_away_goals'].astype(str)
)
group_predictions['winner_icon'] = group_predictions['winning_team'].map(
    {'home': '🏠', 'away': '✈️', 'draw': '🤝'}
)

for grp in sorted(group_predictions['group'].unique()):
    subset = group_predictions[group_predictions['group'] == grp].copy()
    subset = subset[['match_id','home_team','away_team','scoreline',
                     'winner_icon','winning_team','corners',
                     'yellow_cards','red_cards']].reset_index(drop=True)
    subset.columns = ['ID','Home','Away','Score','','Winner',
                      'Corners','Yellows','Reds']
    print(f"\n{'='*75}")
    print(f"  Group {grp}")
    print(f"{'='*75}")
    display(subset)

print(f"\nGroup predictions filled: {len(rows)} matches ✓")


  Group A


,ID,Home,Away,Score,,Winner,Corners,Yellows,Reds
0,1,Mexico,South Africa,1 - 0,🏠,home,7,4,0
1,2,South Korea,Czechia,2 - 1,🏠,home,10,3,0
2,25,Czechia,South Africa,1 - 1,🤝,draw,7,3,0
3,28,Mexico,South Korea,1 - 0,🏠,home,8,3,0
4,53,Czechia,Mexico,0 - 1,✈️,away,9,4,0
5,54,South Africa,South Korea,0 - 1,✈️,away,7,3,0



  Group B


,ID,Home,Away,Score,,Winner,Corners,Yellows,Reds
0,3,Canada,Bosnia & Herzegovina,1 - 0,🏠,home,7,3,0
1,6,Qatar,Switzerland,1 - 4,✈️,away,14,4,0
2,26,Switzerland,Bosnia & Herzegovina,2 - 0,🏠,home,13,4,0
3,27,Canada,Qatar,2 - 0,🏠,home,9,4,0
4,49,Switzerland,Canada,1 - 1,🤝,draw,7,3,0
5,50,Bosnia & Herzegovina,Qatar,2 - 1,🏠,home,12,3,0



  Group C


...


  Group K


,ID,Home,Away,Score,,Winner,Corners,Yellows,Reds
0,21,Portugal,DR Congo,1 - 0,🏠,home,8,4,0
1,24,Uzbekistan,Colombia,0 - 2,✈️,away,11,4,0
2,45,Portugal,Uzbekistan,1 - 0,🏠,home,8,4,0
3,48,Colombia,DR Congo,2 - 0,🏠,home,10,4,0
4,69,Colombia,Portugal,0 - 1,✈️,away,8,3,0
5,70,DR Congo,Uzbekistan,1 - 1,🤝,draw,8,3,0



  Group L


,ID,Home,Away,Score,,Winner,Corners,Yellows,Reds
0,22,England,Croatia,1 - 1,🤝,draw,7,3,0
1,23,Ghana,Panama,0 - 2,✈️,away,10,3,0
2,46,England,Ghana,2 - 0,🏠,home,9,4,0
3,47,Panama,Croatia,1 - 2,✈️,away,11,4,0
4,67,Panama,England,0 - 2,✈️,away,9,3,0
5,68,Croatia,Ghana,2 - 0,🏠,home,10,4,0



Group predictions filled: 72 matches ✓


In [56]:
# ── GROUP STANDINGS + BRACKET ORDER + BEST 3RD (from predicted scorelines) ────

most_likely_standings = {}
third_place_standings = []

for grp in sorted(GROUPS.keys()):
    grp_matches = group_predictions[group_predictions['group'] == grp]
    table = {team: {'Pts': 0, 'W': 0, 'D': 0, 'L': 0, 'GF': 0, 'GA': 0, 'GD': 0}
             for team in GROUPS[grp]}

    for _, m in grp_matches.iterrows():
        h, a   = m['home_team'], m['away_team']
        hg, ag = m['predicted_home_goals'], m['predicted_away_goals']

        table[h]['GF'] += hg;  table[h]['GA'] += ag
        table[a]['GF'] += ag;  table[a]['GA'] += hg
        table[h]['GD']  = table[h]['GF'] - table[h]['GA']
        table[a]['GD']  = table[a]['GF'] - table[a]['GA']

        if hg > ag:
            table[h]['Pts'] += 3; table[h]['W'] += 1; table[a]['L'] += 1
        elif ag > hg:
            table[a]['Pts'] += 3; table[a]['W'] += 1; table[h]['L'] += 1
        else:
            table[h]['Pts'] += 1; table[h]['D'] += 1
            table[a]['Pts'] += 1; table[a]['D'] += 1

    ranked = sorted(table.keys(),
                    key=lambda t: (-table[t]['Pts'], -table[t]['GD'], -table[t]['GF']))

    most_likely_standings[grp] = ranked  # used by _resolve_slot

    third_team = ranked[2]
    third_place_standings.append({
        'group': grp, 'team': third_team,
        'pts':   table[third_team]['Pts'],
        'gd':    table[third_team]['GD'],
        'gf':    table[third_team]['GF'],
    })

# ── Best 3rd ──────────────────────────────────────────────────────────────────
third_place_standings.sort(key=lambda x: (-x['pts'], -x['gd'], -x['gf']))

print("=== Best 3rd Place Teams ===")
best_third_qualifiers = []
for rank, rec in enumerate(third_place_standings, 1):
    status = '✅ QUALIFIES' if rank <= 8 else '❌ out'
    print(f"  {rank:>2}. Group {rec['group']}: {rec['team']:<25} "
          f"Pts={rec['pts']}  GD={rec['gd']:+d}  GF={rec['gf']}  {status}")
    if rank <= 8:
        best_third_qualifiers.append((rec['group'], rec['team']))

assert len(best_third_qualifiers) == 8, \
    f"Expected 8 third-place qualifiers, got {len(best_third_qualifiers)}"
print(f"\n3rd-place qualifiers: {[t for _, t in best_third_qualifiers]}")

# ── Display standings per group ───────────────────────────────────────────────
from IPython.display import display
import pandas as pd

print("\n=== Group Standings ===")
for grp in sorted(GROUPS.keys()):
    grp_matches = group_predictions[group_predictions['group'] == grp]
    table = {team: {'Pts': 0, 'W': 0, 'D': 0, 'L': 0, 'GF': 0, 'GA': 0, 'GD': 0}
             for team in GROUPS[grp]}

    for _, m in grp_matches.iterrows():
        h, a   = m['home_team'], m['away_team']
        hg, ag = m['predicted_home_goals'], m['predicted_away_goals']
        table[h]['GF'] += hg;  table[h]['GA'] += ag
        table[a]['GF'] += ag;  table[a]['GA'] += hg
        table[h]['GD']  = table[h]['GF'] - table[h]['GA']
        table[a]['GD']  = table[a]['GF'] - table[a]['GA']
        if hg > ag:
            table[h]['Pts'] += 3; table[h]['W'] += 1; table[a]['L'] += 1
        elif ag > hg:
            table[a]['Pts'] += 3; table[a]['W'] += 1; table[h]['L'] += 1
        else:
            table[h]['Pts'] += 1; table[h]['D'] += 1
            table[a]['Pts'] += 1; table[a]['D'] += 1

    ranked = sorted(table.keys(),
                    key=lambda t: (-table[t]['Pts'], -table[t]['GD'], -table[t]['GF']))

    rows_s = []
    for pos, team in enumerate(ranked, 1):
        t = table[team]
        medal = {1: '🥇', 2: '🥈', 3: '🥉', 4: ''}.get(pos, '')
        fate  = '✅ Advances' if pos <= 2 else ('🔵 Best 3rd?' if pos == 3 else '❌ Eliminated')
        rows_s.append({
            'Pos':    f"{medal} {pos}",
            'Team':   team,
            'Pts':    t['Pts'],
            'W':      t['W'],
            'D':      t['D'],
            'L':      t['L'],
            'GF':     t['GF'],
            'GA':     t['GA'],
            'GD':     f"{t['GD']:+d}",
            'Status': fate,
        })

    print(f"\n{'='*65}")
    print(f"  Group {grp}")
    print(f"{'='*65}")
    display(pd.DataFrame(rows_s).reset_index(drop=True))

=== Best 3rd Place Teams ===
   1. Group J: Algeria                   Pts=4  GD=+0  GF=2  ✅ QUALIFIES
   2. Group I: Norway                    Pts=4  GD=-1  GF=2  ✅ QUALIFIES
   3. Group E: Côte d'Ivoire             Pts=3  GD=+1  GF=5  ✅ QUALIFIES
   4. Group D: USA                       Pts=3  GD=-1  GF=4  ✅ QUALIFIES
   5. Group L: Panama                    Pts=3  GD=-1  GF=3  ✅ QUALIFIES
   6. Group C: Scotland                  Pts=3  GD=-1  GF=2  ✅ QUALIFIES
   7. Group B: Bosnia & Herzegovina      Pts=3  GD=-2  GF=2  ✅ QUALIFIES
   8. Group G: Iran                      Pts=2  GD=-1  GF=0  ✅ QUALIFIES
   9. Group A: Czechia                   Pts=1  GD=-2  GF=2  ❌ out
  10. Group F: Tunisia                   Pts=1  GD=-3  GF=1  ❌ out
  11. Group K: Uzbekistan                Pts=1  GD=-3  GF=1  ❌ out
  12. Group H: Saudi Arabia              Pts=1  GD=-3  GF=0  ❌ out

3rd-place qualifiers: ['Algeria', 'Norway', "Côte d'Ivoire", 'USA', 'Panama', 'Scotland', 'Bosnia & Herzegovina', 'Ira

,Pos,Team,Pts,W,D,L,GF,GA,GD,Status
0,🥇 1,Mexico,9,3,0,0,3,0,+3,✅ Advances
1,🥈 2,South Korea,6,2,0,1,3,2,+1,✅ Advances
2,🥉 3,Czechia,1,0,1,2,2,4,-2,🔵 Best 3rd?
3,4,South Africa,1,0,1,2,1,3,-2,❌ Eliminated



  Group B


,Pos,Team,Pts,W,D,L,GF,GA,GD,Status
0,🥇 1,Switzerland,7,2,1,0,7,2,+5,✅ Advances
1,🥈 2,Canada,7,2,1,0,4,1,+3,✅ Advances
2,🥉 3,Bosnia & Herzegovina,3,1,0,2,2,4,-2,🔵 Best 3rd?
3,4,Qatar,0,0,0,3,2,8,-6,❌ Eliminated



  Group C


...

,Pos,Team,Pts,W,D,L,GF,GA,GD,Status
0,🥇 1,Argentina,9,3,0,0,4,0,+4,✅ Advances
1,🥈 2,Austria,4,1,1,1,3,3,+0,✅ Advances
2,🥉 3,Algeria,4,1,1,1,2,2,+0,🔵 Best 3rd?
3,4,Jordan,0,0,0,3,1,5,-4,❌ Eliminated



  Group K


,Pos,Team,Pts,W,D,L,GF,GA,GD,Status
0,🥇 1,Portugal,9,3,0,0,3,0,+3,✅ Advances
1,🥈 2,Colombia,6,2,0,1,4,1,+3,✅ Advances
2,🥉 3,Uzbekistan,1,0,1,2,1,4,-3,🔵 Best 3rd?
3,4,DR Congo,1,0,1,2,1,4,-3,❌ Eliminated



  Group L


,Pos,Team,Pts,W,D,L,GF,GA,GD,Status
0,🥇 1,England,7,2,1,0,5,1,+4,✅ Advances
1,🥈 2,Croatia,7,2,1,0,5,2,+3,✅ Advances
2,🥉 3,Panama,3,1,0,2,3,4,-1,🔵 Best 3rd?
3,4,Ghana,0,0,0,3,0,6,-6,❌ Eliminated


In [57]:
# ── KNOCKOUT BRACKET RESOLVER ────────────────────────────────────────

import re as _re

_match_cache = {}   # match_id → {'winner': team, 'loser': team}


# ── _predict_ko helper ────────────────────────────────────────────────────────
def _predict_ko(home_team, away_team):
    """
    Predict a knockout match (neutral venue).
    Returns dict with hg/ag/corners/yellow_cards/winner_side/winner_team/loser_team/penalties.
    Returns None if either team is TBD.
    """
    if home_team == 'TBD' or away_team == 'TBD':
        return None
    pred = predict_match_adj(home_team, away_team, neutral=True)
    (hg, ag), _ = best_scoreline_for_points(pred, include_winner_bonus=True,
                                             winner_bonus_pts=20)

    if pred['p_home_win'] >= pred['p_away_win']:
        winner_side, winner_team, loser_team = 'home', home_team, away_team
    else:
        winner_side, winner_team, loser_team = 'away', away_team, home_team

    lam_total = pred['lambda_h'] + pred['lambda_a']
    corners   = max(8, min(14, round(10.8 * lam_total / 2.5)))
    pen_gap   = abs(pred['p_home_win'] - pred['p_away_win'])
    penalties = (hg == ag) and (pen_gap < 0.12)

    return dict(hg=hg, ag=ag, corners=corners, yellow_cards=4, red_cards=0,
                winner_side=winner_side, winner_team=winner_team,
                loser_team=loser_team, penalties=penalties)

# ── PRE-ASSIGN Best-3rd slots ─────────────────
from scipy.optimize import linear_sum_assignment

best3rd_slots = knockout_slots[
    knockout_slots['slot_home'].str.contains('Best 3rd', na=False) |
    knockout_slots['slot_away'].str.contains('Best 3rd', na=False)
].copy()

# Collect unique Best-3rd slot descriptions in match_id order
_seen = set()
_slot_list = []   # [(slot_text, eligible_groups)]
for _, row in best3rd_slots.sort_values('match_id').iterrows():
    for col in ['slot_home', 'slot_away']:
        slot = str(row[col])
        if 'Best 3rd' in slot and slot not in _seen:
            _seen.add(slot)
            m = _re.search(r'\(Groups ([^)]+)\)', slot)
            eligible = m.group(1).split('/') if m else []
            _slot_list.append((slot, eligible))

n = len(_slot_list)
assert n == len(best_third_qualifiers), f"Slot count {n} != qualifier count {len(best_third_qualifiers)}"

# Build cost matrix: rows = slots, cols = qualifiers (ordered by probability)
_cost = np.full((n, n), 1e9)
for i, (slot_text, eligible) in enumerate(_slot_list):
    for j, (grp, team) in enumerate(best_third_qualifiers):
        if grp in eligible:
            _cost[i, j] = j   # lower j = higher probability = preferred

row_ind, col_ind = linear_sum_assignment(_cost)

_best3rd_map = {}
for i, j in zip(row_ind, col_ind):
    slot_text, _ = _slot_list[i]
    grp, team     = best_third_qualifiers[j]
    _best3rd_map[slot_text] = team

print(f"Best-3rd assignments ({len(_best3rd_map)} slots):")
for slot, team in _best3rd_map.items():
    print(f"  {slot}  →  {team}")

assert 'TBD' not in _best3rd_map.values(), "Some Best-3rd slots still unassigned!"


# ── Slot resolver ─────────────────────────────────────────────────────────────
def _resolve_slot(slot):
    slot = str(slot).strip()

    if slot.startswith('Winner Group '):
        grp = slot.replace('Winner Group ', '').strip()
        return most_likely_standings[grp][0]

    if slot.startswith('Runner-up Group '):
        grp = slot.replace('Runner-up Group ', '').strip()
        return most_likely_standings[grp][1]

    if 'Best 3rd' in slot:
        return _best3rd_map.get(slot, 'TBD')

    if slot.startswith('Winner Match '):
        mid = int(slot.replace('Winner Match ', '').strip())
        return _match_cache.get(mid, {}).get('winner', 'TBD')

    if slot.startswith('Loser Match '):
        mid = int(slot.replace('Loser Match ', '').strip())
        return _match_cache.get(mid, {}).get('loser', 'TBD')

    return slot


print("\nBracket resolver ready.")

Best-3rd assignments (8 slots):
  Best 3rd (Groups A/B/C/D/F)  →  USA
  Best 3rd (Groups C/D/F/G/H)  →  Scotland
  Best 3rd (Groups C/E/F/H/I)  →  Norway
  Best 3rd (Groups E/H/I/J/K)  →  Algeria
  Best 3rd (Groups A/E/H/I/J)  →  Côte d'Ivoire
  Best 3rd (Groups B/E/F/I/J)  →  Bosnia & Herzegovina
  Best 3rd (Groups E/F/G/I/J)  →  Iran
  Best 3rd (Groups D/E/I/J/L)  →  Panama

Bracket resolver ready.


## 🏆 Knockout stage predictions

In [58]:

# ── KNOCKOUT STAGE PREDICTIONS ────────────────────────────────────────────────
# Process matches in ascending match_id order so that "Winner Match N" slots
# for later rounds can look up earlier rounds in _match_cache.
# ─────────────────────────────────────────────────────────────────────────────

knockout_predictions = knockout_slots.copy()
knockout_predictions['predicted_home_team']  = None
knockout_predictions['predicted_away_team']  = None
knockout_predictions['predicted_home_goals'] = None
knockout_predictions['predicted_away_goals'] = None
knockout_predictions['corners']              = None
knockout_predictions['yellow_cards']         = None
knockout_predictions['red_cards']            = None
knockout_predictions['match_winner']         = None
knockout_predictions['penalties']            = None

for _, slot_row in knockout_slots.sort_values('match_id').iterrows():
    mid       = slot_row['match_id']
    home_team = _resolve_slot(slot_row['slot_home'])
    away_team = _resolve_slot(slot_row['slot_away'])
    result    = _predict_ko(home_team, away_team)

    loc = knockout_predictions.index[knockout_predictions['match_id'] == mid][0]
    knockout_predictions.at[loc, 'predicted_home_team'] = home_team
    knockout_predictions.at[loc, 'predicted_away_team'] = away_team

    if result:
        knockout_predictions.at[loc, 'predicted_home_goals'] = int(result['hg'])
        knockout_predictions.at[loc, 'predicted_away_goals'] = int(result['ag'])
        knockout_predictions.at[loc, 'corners']              = int(result['corners'])
        knockout_predictions.at[loc, 'yellow_cards']         = int(result['yellow_cards'])
        knockout_predictions.at[loc, 'red_cards']            = 0
        knockout_predictions.at[loc, 'match_winner']         = result['winner_side']
        knockout_predictions.at[loc, 'penalties']            = bool(result['penalties'])
        _match_cache[mid] = {'winner': result['winner_team'], 'loser': result['loser_team']}

print("=== Knockout Predictions ===")
print(knockout_predictions[['match_id','round','predicted_home_team','predicted_away_team',
                             'predicted_home_goals','predicted_away_goals',
                             'match_winner','penalties']].to_string(index=False))

print("\n=== Final sanity: no None values remaining? ===")
null_counts = knockout_predictions[['predicted_home_team','predicted_away_team',
                                     'predicted_home_goals','predicted_away_goals',
                                     'corners','yellow_cards','red_cards',
                                     'match_winner','penalties']].isnull().sum()
print(null_counts.to_string())


=== Knockout Predictions ===
 match_id               round predicted_home_team  predicted_away_team predicted_home_goals predicted_away_goals match_winner penalties
       73         Round of 32         South Korea               Canada                    0                    0         away      True
       74         Round of 32              Brazil                Japan                    2                    1         home     False
       75         Round of 32             Germany                  USA                    2                    1         home     False
       76         Round of 32         Netherlands              Morocco                    1                    2         away     False
       77         Round of 32             Ecuador              Senegal                    1                    0         home     False
       78         Round of 32              France             Scotland                    2                    0         home     False
       79         R

In [59]:
# ── Display ────────────────────────────────────────────────────────────────────

import pandas as pd
from IPython.display import display

round_order = ['Round of 32','Round of 16','Quarter-final','Semi-final',
               'Third-place playoff','Final']
round_cat   = pd.CategoricalDtype(categories=round_order, ordered=True)
knockout_predictions['round'] = knockout_predictions['round'].astype(round_cat)

display_cols = ['match_id','round','slot_home','slot_away',
                'predicted_home_team','predicted_away_team',
                'predicted_home_goals','predicted_away_goals',
                'match_winner','penalties','corners','yellow_cards']

for rnd in round_order:
    subset = knockout_predictions[knockout_predictions['round'] == rnd][display_cols]
    if subset.empty:
        continue

    # Build readable scoreline column
    subset = subset.copy()
    subset['scoreline'] = (
        subset['predicted_home_goals'].astype(str) + ' - ' +
        subset['predicted_away_goals'].astype(str)
    )
    subset['match'] = subset['predicted_home_team'] + '  vs  ' + subset['predicted_away_team']
    subset['pens?'] = subset['penalties'].map({True: '✓', False: ''})

    print(f"\n{'='*90}")
    print(f"  {rnd}")
    print(f"{'='*90}")

    show = subset[['match_id','slot_home','slot_away','match','scoreline','match_winner','pens?','corners','yellow_cards']].copy()
    show.columns = ['ID','Slot Home','Slot Away','Match','Score','Winner','Pens','Corners','Yellows']
    show = show.reset_index(drop=True)
    display(show)

# ── Null check ─────────────────────────────────────────────────────────────────
print("\n=== Null check ===")
null_counts = knockout_predictions[['predicted_home_team','predicted_away_team',
                                    'predicted_home_goals','predicted_away_goals',
                                    'corners','yellow_cards','red_cards',
                                    'match_winner','penalties']].isnull().sum()
print(null_counts[null_counts > 0].to_string() if null_counts.any() else "No nulls ✓")

# ── Cache trace (helps debug bracket chain) ────────────────────────────────────
print("\n=== Match cache (bracket chain) ===")
for mid_c, result_c in sorted(_match_cache.items()):
    print(f"  Match {mid_c}: winner={result_c['winner']:<25} loser={result_c['loser']}")


  Round of 32


,ID,Slot Home,Slot Away,Match,Score,Winner,Pens,Corners,Yellows
0,73,Runner-up Group A,Runner-up Group B,South Korea vs Canada,0 - 0,away,✓,8,4
1,74,Winner Group C,Runner-up Group F,Brazil vs Japan,2 - 1,home,,12,4
2,75,Winner Group E,Best 3rd (Groups A/B/C/D/F),Germany vs USA,2 - 1,home,,14,4
3,76,Winner Group F,Runner-up Group C,Netherlands vs Morocco,1 - 2,away,,13,4
4,77,Runner-up Group E,Runner-up Group I,Ecuador vs Senegal,1 - 0,home,,8,4
5,78,Winner Group I,Best 3rd (Groups C/D/F/G/H),France vs Scotland,2 - 0,home,,10,4
6,79,Winner Group A,Best 3rd (Groups C/E/F/H/I),Mexico vs Norway,1 - 1,home,,8,4
7,80,Winner Group L,Best 3rd (Groups E/H/I/J/K),England vs Algeria,0 - 0,home,✓,8,4
8,81,Winner Group G,Best 3rd (Groups A/E/H/I/J),Belgium vs Côte d'Ivoire,2 - 1,home,,14,4
9,82,Winner Group D,Best 3rd (Groups B/E/F/I/J),Türkiye vs Bosnia & Herzegovina,2 - 1,home,,14,4



  Round of 16


,ID,Slot Home,Slot Away,Match,Score,Winner,Pens,Corners,Yellows
0,89,Winner Match 73,Winner Match 75,Canada vs Germany,0 - 1,away,,8,4
1,90,Winner Match 74,Winner Match 77,Brazil vs Ecuador,1 - 1,home,✓,8,4
2,91,Winner Match 76,Winner Match 78,Morocco vs France,1 - 2,away,,11,4
3,92,Winner Match 79,Winner Match 80,Mexico vs England,1 - 1,home,✓,8,4
4,93,Winner Match 83,Winner Match 84,Colombia vs Spain,1 - 2,away,,14,4
5,94,Winner Match 81,Winner Match 82,Belgium vs Türkiye,2 - 1,home,,14,4
6,95,Winner Match 86,Winner Match 88,Paraguay vs Portugal,0 - 1,away,,8,4
7,96,Winner Match 85,Winner Match 87,Switzerland vs Argentina,1 - 2,away,,12,4



  Quarter-final


...


  Third-place playoff


,ID,Slot Home,Slot Away,Match,Score,Winner,Pens,Corners,Yellows
0,103,Loser Match 101,Loser Match 102,Brazil vs France,1 - 2,away,,14,4



  Final


,ID,Slot Home,Slot Away,Match,Score,Winner,Pens,Corners,Yellows
0,104,Winner Match 101,Winner Match 102,Spain vs Argentina,1 - 2,away,,12,4



=== Null check ===
No nulls ✓

=== Match cache (bracket chain) ===
  Match 73: winner=Canada                    loser=South Korea
  Match 74: winner=Brazil                    loser=Japan
  Match 75: winner=Germany                   loser=USA
  Match 76: winner=Morocco                   loser=Netherlands
  Match 77: winner=Ecuador                   loser=Senegal
  Match 78: winner=France                    loser=Scotland
  Match 79: winner=Mexico                    loser=Norway
  Match 80: winner=England                   loser=Algeria
  Match 81: winner=Belgium                   loser=Côte d'Ivoire
  Match 82: winner=Türkiye                   loser=Bosnia & Herzegovina
  Match 83: winner=Colombia                  loser=Croatia
  Match 84: winner=Spain                     loser=Austria
  Match 85: winner=Switzerland               loser=Iran
  Match 86: winner=Paraguay                  loser=Egypt
  Match 87: winner=Argentina                 loser=Uruguay
  Match 88: winner=Portugal    